# Creating Custom Algorithms in SocialJax (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/SocialJax/blob/main/tutorials/02_custom_algorithm_colab.ipynb)

This tutorial shows you how to create your own multi-agent reinforcement learning algorithms using SocialJax's modular architecture.

## Overview

SocialJax provides a clean abstraction for implementing new algorithms:

1. **BaseAlgorithm**: Abstract base class with required methods
2. **Algorithm State**: JAX-compatible state management
3. **Registry System**: Register your algorithm for easy access

Let's implement a simple **REINFORCE** algorithm as an example!

## 0. Colab Setup

In [ ]:
# @title Colab Setup { display-mode: "form" }

import os
import sys

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    print("Installing SocialJax on Google Colab...")
    
    !git clone https://github.com/cooperativex/SocialJax.git
    %cd SocialJax
    
    !pip install -q "numpy<2.0"
    !pip install -q -r requirements.txt
    !pip install -q jax[cuda11_pip]==0.4.30 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    
    sys.path.insert(0, '/content/SocialJax/socialjax')
    print("\u2705 SocialJax installed successfully!")
else:
    print("Not running on Colab. Make sure SocialJax is installed locally.")
    sys.path.insert(0, './socialjax')

In [ ]:
# @title Check GPU Availability
import jax
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. Import Required Components

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax import struct
import optax

# SocialJax imports
from socialjax.core import BaseAlgorithm, AlgorithmState
from socialjax.algorithms import register_algorithm, get_algorithm
from socialjax.networks import create_network

## 2. Define Configuration

First, let's define the hyperparameters for our algorithm.

In [ ]:
# Configuration dictionary
REINFORCE_CONFIG = {
    'learning_rate': 0.001,
    'gamma': 0.99,  # Discount factor
    'entropy_coef': 0.01,  # Entropy bonus coefficient
    'max_grad_norm': 0.5,  # Gradient clipping
}

print("REINFORCE Configuration:")
for key, value in REINFORCE_CONFIG.items():
    print(f"  {key}: {value}")

## 3. Define Algorithm State

Algorithm state is a JAX-compatible dataclass that stores trainable parameters and training state.

In [ ]:
@struct.dataclass
class REINFORCEAlgorithmState(AlgorithmState):
    """State for REINFORCE algorithm.
    
    Inherits from AlgorithmState which provides:
    - params: Neural network parameters
    - optimizer_state: Optimizer state (e.g., momentum)
    - rng: JAX random key
    - timestep: Current training timestep
    
    We can add additional fields specific to REINFORCE:
    """
    episode_count: int = 0  # Number of episodes completed

## 4. Define the Algorithm Class

Now let's implement the main algorithm class with all required methods.

In [ ]:
@register_algorithm("reinforce")
class REINFORCEAlgorithm(BaseAlgorithm):
    """Simple REINFORCE (Policy Gradient) algorithm.
    
    This is a basic policy gradient method that:
    1. Collects complete episodes
    2. Computes returns using Monte Carlo estimation
    3. Updates policy using the policy gradient theorem
    """
    
    def __init__(
        self,
        observation_space,
        action_space,
        config=None,
    ):
        super().__init__(observation_space, action_space, config)
        self.config = {**REINFORCE_CONFIG, **(config or {})}
        
        # Create network and optimizer
        self._network = self._build_network()
        self._optimizer = self._build_optimizer()
    
    def _build_network(self):
        """Build the policy network."""
        # Use SocialJax's network factory
        return create_network(
            "cnn_small",  # Use CNN for visual observations
            action_dim=self.action_space.n,
        )
    
    def _build_optimizer(self):
        """Build the optimizer."""
        return optax.chain(
            optax.clip_by_global_norm(self.config['max_grad_norm']),
            optax.adam(self.config['learning_rate']),
        )
    
    def init_state(self, key) -> REINFORCEAlgorithmState:
        """Initialize algorithm state.
        
        Args:
            key: JAX random key
            
        Returns:
            Initial algorithm state
        """
        # Initialize network parameters
        dummy_obs = jnp.zeros(self.observation_space.shape)
        params = self._network.init(key, dummy_obs)
        
        # Initialize optimizer state
        optimizer_state = self._optimizer.init(params)
        
        return REINFORCEAlgorithmState(
            params=params,
            optimizer_state=optimizer_state,
            rng=key,
            timestep=0,
            episode_count=0,
        )
    
    def compute_action(
        self,
        state: REINFORCEAlgorithmState,
        obs,
        deterministic: bool = False,
    ):
        """Compute action from observation.
        
        Args:
            state: Current algorithm state
            obs: Observation from environment
            deterministic: If True, return argmax action
            
        Returns:
            Tuple of (action, log_prob, new_state)
        """
        key, action_key = jax.random.split(state.rng)
        
        # Get action distribution
        logits = self._network.apply(state.params, obs)
        probs = jax.nn.softmax(logits, axis=-1)
        
        if deterministic:
            action = jnp.argmax(probs, axis=-1)
            log_prob = jnp.log(probs[action] + 1e-8)
        else:
            # Sample from distribution
            action = jax.random.categorical(action_key, logits)
            log_prob = jax.nn.log_softmax(logits)[action]
        
        new_state = state.replace(rng=key)
        return action, log_prob, new_state
    
    def compute_returns(self, rewards, dones, gamma):
        """Compute discounted returns using Monte Carlo.
        
        Args:
            rewards: Array of rewards
            dones: Array of done flags
            gamma: Discount factor
            
        Returns:
            Array of returns
        """
        def scan_fn(carry, x):
            ret, = carry
            reward, done = x
            ret = reward + gamma * ret * (1 - done)
            return (ret,), ret
        
        _, returns = jax.lax.scan(
            scan_fn,
            (0.0,),
            (rewards, dones),
            reverse=True,
        )
        return returns
    
    def update(self, state: REINFORCEAlgorithmState, batch):
        """Update policy using collected experience.
        
        Args:
            state: Current algorithm state
            batch: Dictionary containing:
                - observations: Array of observations
                - actions: Array of actions
                - rewards: Array of rewards
                - dones: Array of done flags
                - log_probs: Array of log probabilities
                
        Returns:
            Tuple of (new_state, metrics)
        """
        obs = batch['observations']
        actions = batch['actions']
        rewards = batch['rewards']
        dones = batch['dones']
        
        # Compute returns
        returns = self.compute_returns(rewards, dones, self.config['gamma'])
        
        # Normalize returns for stability
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        def loss_fn(params):
            """Compute policy gradient loss."""
            # Get action logits
            logits = self._network.apply(params, obs)
            log_probs = jax.nn.log_softmax(logits)
            
            # Select log probs for taken actions
            selected_log_probs = log_probs[jnp.arange(len(actions)), actions]
            
            # Policy gradient loss (negative because we maximize)
            policy_loss = -jnp.mean(selected_log_probs * returns)
            
            # Entropy bonus for exploration
            probs = jax.nn.softmax(logits)
            entropy = -jnp.sum(probs * log_probs, axis=-1).mean()
            entropy_loss = -self.config['entropy_coef'] * entropy
            
            total_loss = policy_loss + entropy_loss
            return total_loss, {
                'policy_loss': policy_loss,
                'entropy': entropy,
                'total_loss': total_loss,
            }
        
        # Compute gradients and update
        (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
        updates, new_optimizer_state = self._optimizer.update(grads, state.optimizer_state)
        new_params = optax.apply_updates(state.params, updates)
        
        new_state = state.replace(
            params=new_params,
            optimizer_state=new_optimizer_state,
            timestep=state.timestep + len(rewards),
        )
        
        return new_state, metrics

## 5. Test the Algorithm

Let's verify that our algorithm is registered and can be created.

In [ ]:
from socialjax.algorithms import list_algorithms

# Check that our algorithm is registered
print("Registered algorithms:")
for algo in list_algorithms():
    print(f"  - {algo}")

# Verify REINFORCE is in the list
assert 'reinforce' in list_algorithms(), "REINFORCE should be registered!"
print("\nREINFORCE algorithm successfully registered!")

## 6. Use the Algorithm

Now let's use our custom algorithm with SocialJax's Trainer.

In [ ]:
from socialjax import make

# Create environment
env = make('coin_game', num_agents=3)

# Get algorithm class and create instance
REINFORCE = get_algorithm('reinforce')

# Create algorithm instance
algo = REINFORCE(
    observation_space=env.observation_space(),
    action_space=env.action_space(),
    config={'learning_rate': 0.0005},
)

# Initialize state
key = jax.random.PRNGKey(42)
state = algo.init_state(key)

print(f"Algorithm: REINFORCE")
print(f"State type: {type(state).__name__}")
print(f"Timestep: {state.timestep}")

## 7. Test Action Computation

In [ ]:
# Reset environment
obs, env_state = env.reset(jax.random.PRNGKey(0))

# Compute action for first agent
action, log_prob, state = algo.compute_action(state, obs[0])

print(f"Observation shape: {obs[0].shape}")
print(f"Action: {action}")
print(f"Log probability: {log_prob:.4f}")

## 8. Algorithm Best Practices

When implementing custom algorithms, follow these guidelines:

### Required Methods

1. **`_build_network()`**: Create your neural network
2. **`_build_optimizer()`**: Create your optimizer
3. **`init_state()`**: Initialize algorithm state
4. **`compute_action()`**: Compute actions from observations
5. **`update()`**: Update policy from experience

### Optional Methods

- **`compute_value()`**: Compute value estimates (for actor-critic methods)
- **`save()`** / **`load()`**: Custom checkpoint handling

### JAX Compatibility

- Use `@struct.dataclass` for state classes
- Use `state.replace()` to update immutable state
- Use `jax.lax.scan` for loops (for JIT compatibility)
- Avoid Python control flow in JAX functions

In [ ]:
# Example: Proper state update pattern
print("State update pattern:")
print("""\n# \u274C Don't do this (mutates state)
state.timestep = state.timestep + 1

# \u2705 Do this (creates new state)
new_state = state.replace(timestep=state.timestep + 1)
""")

## 9. Advanced: Adding Value Function

For actor-critic algorithms, implement `compute_value()`:

In [ ]:
def compute_value(self, state, obs):
    """Compute value estimate for state.
    
    Args:
        state: Algorithm state
        obs: Observation
        
    Returns:
        Value estimate
    """
    # Assuming network outputs (logits, value)
    _, value = self._network.apply(state.params, obs)
    return value

print("Value function signature defined for actor-critic methods.")

## Summary

In this tutorial, you learned:

1. **BaseAlgorithm structure**: Required and optional methods
2. **State management**: Using `@struct.dataclass` for JAX compatibility
3. **Registration**: Using `@register_algorithm` decorator
4. **Implementation patterns**: Best practices for JAX-based algorithms

## Next Steps

- **Tutorial 3**: Creating custom network architectures
- **Tutorial 4**: Using callbacks for logging and monitoring
- **Tutorial 5**: Advanced configuration options